# Лабораторная 1. Как движутся данные

Ваш первый рабочий день в интернет-магазине. К вам подходит аналитик:

> Мне нужен отчёт: сколько заказов в день, на какую сумму и сколько мы платим
> за доставку. Данные где-то есть, спроси у ребят.

«У ребят» обнаруживается девять файлов: ночная выгрузка из систем магазина.
Заказы лежат отдельно от товарных позиций, позиции – отдельно от платежей,
а отзывы собирает отдельный сервис опросов. Ни один файл сам по себе на вопрос
аналитика не отвечает, а если соединять таблицы, не разобравшись в их
структуре, легко получить убедительный, но неверный результат.

Просьба звучала как «сделай отчёт». На деле это работа инженера данных:
достать, сложить и не соврать в цифрах. С этого отчёта начинается система,
которую мы будем строить весь курс.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **Приём данных – ingestion** | Шаг, на котором данные попадают из источника к нам. |
| **Сырой слой – raw layer** | Копия данных в том виде, в каком они приехали, без исправлений. |
| **Зерно таблицы – grain** | Ответ на вопрос «что означает одна строка этой таблицы». |
| **Соединение таблиц – JOIN** | Сопоставление строк двух таблиц по общему ключу. |
| **Размножение строк – fan-out** | Когда после соединения одна строка слева превращается в несколько. |
| **Идемпотентность – idempotency** | Свойство операции давать тот же результат при повторном запуске. |

Эти шесть слов пока можно просто запомнить на слух: каждое из них дальше
появится на конкретных данных.

## Подготовка

Ячейка готовит окружение и данные. Её достаточно запустить.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import pandas as pd
import paths
from course_visuals import show_visual
from get_data import get_olist

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

RAW = get_olist()

## Путь, который мы сегодня пройдём

Сегодняшняя задача умещается в четыре блока: забрать файлы, сохранить их не
трогая, посчитать нужное, отдать результат тому, кто его просил. Дальше по
курсу между этими блоками появятся новые, но сам путь не изменится.

In [ ]:
show_visual("flow")

## Что нам прислали

Прежде чем что-то читать в память, полезно посмотреть на файлы как на файлы:
сколько их, сколько весят, какие колонки в заголовке. На больших выгрузках эта
минута экономит часы – бывает, что нужной колонки в файле просто нет, и узнать
об этом лучше сразу.

In [ ]:
def header_of(path) -> str:
    """Первая строка файла – это его заголовок с именами колонок."""
    with path.open(encoding="utf-8") as f:
        return f.readline().strip()[:58] + "…"


files = sorted(RAW.glob("*.csv"))
pd.DataFrame({
    "файл": [f.name for f in files],
    "размер, МБ": [round(f.stat().st_size / 1024 ** 2, 2) for f in files],
    "начало заголовка": [header_of(f) for f in files],
})

Перед нами девять связанных таблиц одного набора данных. Связи между ними
выглядят так:

In [ ]:
show_visual("olist")

Справочники товаров и продавцов на схеме опущены, чтобы не загромождать её:
сегодня нас интересуют три ветки справа от заказа. У одного заказа бывает
несколько товарных позиций, несколько платёжных операций и иногда несколько
отзывов. Чуть дальше увидим, во что это превращается в цифрах.

Данные у нас лежат в файлах, но источники бывают разные, и у каждого своя
больная точка:

| Откуда берём | Пример | Чем неудобен |
|---|---|---|
| Рабочая база приложения | PostgreSQL с заказами | тяжёлый запрос к ней тормозит сам магазин |
| Файлы от партнёров | выгрузки поставщиков | структура меняется без предупреждения |
| Внешний интерфейс – API | служба доставки | ограничения на частоту запросов, таймауты |
| Поток событий | клики на сайте | объём и порядок прихода |
| Файлы-объекты | фотографии товаров | тяжёлые сами по себе, лёгкие в описании |

По упорядоченности данные делят на три вида: строгая таблица с фиксированными
колонками; гибкая структура вроде JSON или логов, где набор полей может
меняться от записи к записи; и свободные данные – картинки, звук, текст отзыва.
Деление это не про расширение файла: JSON с одинаковыми полями у всех записей
упорядочен лучше, чем CSV, в колонке которого лежит свободный текст со
сломанными кавычками. У нас сегодня первый вид в чистом виде.

## Кому звонят, когда в отчёте неверное число

Пока компания маленькая, отдельный человек для данных не нужен: есть база
приложения, аналитик пишет к ней запросы, все довольны. Потребность возникает
в конкретный момент – когда источников становится больше одного и они не
согласованы, когда аналитический запрос кладёт рабочую базу, когда отчёт нужен
каждое утро, а не «когда Петя запустит скрипт», и, чаще всего, когда кто-то
один раз посчитал выручку неправильно и доверие к цифрам пропало.

Дальше роли расходятся по зоне ответственности:

| Роль | За что отвечает головой |
|---|---|
| Инженер данных – **Data Engineer** | данные доехали целиком, не задвоились и успели к утру |
| Аналитик – **Data Analyst** | вопрос понят правильно, а совпадение не выдано за причину |
| **Data Scientist** | закономерность найдена и проверена, а не померещилась на выборке |
| **ML Engineer** | модель крутится под нагрузкой и не утаскивает сервис за собой |

В маленькой компании это один человек, в большой – четыре отдела, но граница
проходит по ответственности, а не по инструментам. Инженеру данных звонят,
когда в отчёте неверное число: он отвечает не за смысл показателя, а за то,
что показатель посчитан по тем данным, которые действительно были.

## Почему исходные файлы держат отдельно

Первый шаг любой системы – забрать данные из источника и положить у себя. Этот
шаг называют **приёмом данных (ingestion)**, а слой, куда данные попадают, –
**сырым (raw layer)**: здесь они лежат максимально близко к тому виду, в каком
приехали.

Соблазн поправить данные прямо на входе велик, но в сыром слое их стараются не
менять, и в нашем учебном проекте мы будем считать этот слой неизменяемым.
Причин две, и обе проверяются на практике быстро. Источник не обязан отдать те
же данные повторно: партнёр перезаливает файл поверх старого, внешний сервис
хранит историю за тридцать дней, рабочая база помнит текущее состояние заказа,
а не его историю. И вторая: когда в расчётах найдётся ошибка – а она
найдётся, – пересчитать прошлые периоды можно будет только из сырого слоя.

Поэтому исправления живут в следующем слое, а сырой остаётся архивом.
В третьей работе мы построим этот следующий слой целиком.

In [ ]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

print(f"заказы            : {len(orders):>7,} строк, {orders.shape[1]} колонок")
print(f"позиции заказов   : {len(items):>7,} строк, {items.shape[1]} колонок")
print(f"покупатели        : {len(customers):>7,} строк, {customers.shape[1]} колонок")
orders.head(3)

In [ ]:
pd.DataFrame({
    "тип": orders.dtypes.astype(str),
    "пропусков": orders.isna().sum(),
    "% пропусков": (orders.isna().mean() * 100).round(1),
    "уникальных": orders.nunique(),
})

В этой таблице видно три вещи, каждая из которых аукнется дальше.

Все даты приехали текстом. CSV не хранит типы, в нём нет ничего, кроме
символов, поэтому любой, кто читает файл, угадывает типы заново. Сравнение дат
в виде строк даст ответ, но не тот: `'2018-9-5'` окажется больше, чем
`'2018-12-1'`, потому что сравниваются символы, а не моменты времени.

Пропуски в датах доставки есть, и это нормально: отменённый заказ никогда не
был доставлен, пустая дата – честное описание реальности. Дефект выглядит
иначе: статус «доставлен», а даты доставки нет. Такие строки в данных тоже
есть, ловить их мы будем в третьей работе. Пропуск сам по себе не ошибка,
ошибкой его делает противоречие с соседней колонкой.

И третье: `order_id` уникален. Одна строка этой таблицы – ровно один заказ.

## Что означает одна строка?

Вопрос выглядит детским ровно до того дня, когда из него вырастает расхождение
в отчёте, у которого есть цена в деньгах: в нашем случае – в бразильских
реалах, в вашем – в чём придётся объясняться перед финансовым директором.
У этого свойства таблицы есть название – **зерно (grain)**.

В наших данных три таблицы с разным зерном. В `orders` строка – это заказ.
В `order_items` строка – одна товарная позиция, и у крупного заказа их может
быть два десятка. В `payments` строка – одна платёжная операция: часть суммы
картой, часть сертификатом – уже две строки на один заказ.

Что получится, если соединить заказы с позициями напрямую:

In [ ]:
naive = orders.merge(items, on="order_id", how="inner")

print(f"заказов в исходной таблице     : {len(orders):>7,}")
print(f"строк после соединения          : {len(naive):>7,}")
print(f"уникальных заказов в результате : {naive['order_id'].nunique():>7,}")
print()
print(f"если посчитать заказы как число строк, ошибка составит "
      f"{(len(naive) / naive['order_id'].nunique() - 1) * 100:.1f}%")

In [ ]:
# заказ, у которого больше всего товарных позиций
example = items["order_id"].value_counts().idxmax()

naive.loc[naive["order_id"] == example,
          ["order_id", "order_item_id", "product_id", "price", "freight_value"]]

pandas не ошибся. Он сделал ровно то соединение, которое мы написали, и вернул
все подходящие пары строк. Ошиблись мы раньше – когда не посмотрели, что
означает одна строка каждой таблицы.

Само по себе это ещё не поломка: мы получили таблицу другого зерна, где строка
означает позицию, а не заказ. Такое превращение называют **размножением строк
(fan-out)**. Поломка начинается, когда по такой таблице считают что-то уровня
заказа.

Тонкость, которую стоит проговорить отдельно: страдает не та таблица, которую
размножили, а та, которую размножили вместе с ней. Сумма `price` по позициям
останется верной – каждая позиция попала в результат один раз. А вот стоимость
доставки заказа или его итоговая сумма, если бы они хранились в таблице
заказов, продублировались бы столько раз, сколько позиций в заказе.

Отсюда правило, которое дальше в курсе не изменится: прежде чем соединять,
приведите правую таблицу к зерну левой – сначала свёртка, потом соединение.

## Раз в сутки или сразу

Второе решение, которое принимают в самом начале, – как обрабатывать данные.

**Пакетная обработка (batch processing)**: у нас есть конечная порция данных,
накопившаяся за период, мы берём её целиком и считаем. Процесс имеет начало
и конец, его можно повторить на тех же входных данных и получить тот же
результат.

**Потоковая обработка (streaming)**: данные продолжают поступать, и система
обрабатывает их по мере появления. Процесс не заканчивается; вместо «всех
данных» есть только то, что пришло к этому моменту, и приходится решать, что
считать последним часом, если события опаздывают, и что делать с событием,
которое пришло дважды.

Из этой разницы следует остальное. Пакетную обработку проще перезапустить
и проверить, поэтому ей отдают отчётность и подготовку данных для обучения
моделей. Потоковую выбирают там, где допустимая задержка измеряется секундами
или минутами: остановить подозрительный платёж до списания денег пакетной
обработкой раз в сутки не получится.

Наш сегодняшний отчёт – пакетная обработка в чистом виде.

## Три шага, через которые удобно смотреть на обработку

Дальше по курсу почти все наши процессы будут раскладываться на три действия:
забрать данные из источника, привести их к нужному виду и зерну, положить туда,
где их прочитает потребитель. Такую цепочку называют **конвейером обработки
(data pipeline)**.

Слово «конвейер» здесь не метафора: важно, что его можно запустить повторно
и получить тот же результат, не вспоминая, в каком порядке нажимать ячейки.
Код, который работает только в руках автора и только с первого раза, – пока
черновик. К пятой работе наш черновик станет расписанием с журналом запусков
и восстановлением после сбоя, а сейчас он умещается в три функции.

Результат мы сохраним в файл формата Parquet. Пока воспринимайте его просто как
формат, в котором удобно хранить таблицу; чем он лучше CSV для аналитических
данных, разберём в следующей работе.

In [ ]:
from pathlib import Path


def extract_orders(raw_dir: Path) -> pd.DataFrame:
    """Забрать: читаем из источника как есть."""
    return pd.read_csv(raw_dir / "olist_orders_dataset.csv")


def transform_daily_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Преобразовать: приводим тип даты и сворачиваем до зерна «один день»."""
    out = df.copy()
    out["order_purchase_timestamp"] = pd.to_datetime(out["order_purchase_timestamp"])
    out["order_date"] = out["order_purchase_timestamp"].dt.date
    daily = out.groupby("order_date", as_index=False).agg(
        n_orders=("order_id", "nunique"))
    return daily.sort_values("order_date").reset_index(drop=True)


def load_parquet(df: pd.DataFrame, path: Path) -> Path:
    """Сохранить: кладём результат туда, где его найдёт потребитель."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    return path


demo = transform_daily_orders(extract_orders(RAW))
load_parquet(demo, paths.STAGING / "lab01_demo_daily.parquet")

print(f"дней в результате: {len(demo)}, заказов всего: {demo['n_orders'].sum():,}")
print(f"сходится с исходной таблицей: {demo['n_orders'].sum() == len(orders)}")
demo.tail(5)

Последняя строка вывода стоит того, чтобы войти в привычку: расчёт
заканчивается сверкой с источником. Столько же заказов, сколько было, – или
уже нет, и тогда лучше узнать об этом сейчас, а не из чужого дашборда. Дальше в курсе
такие сверки станут автоматическими проверками, которые умеют останавливать
обработку.

## Что ломается в этом месте

Два коротких опыта. Оба выглядят мелочью и оба вырастают в отдельные темы
следующих работ.

**Сохранили результат и прочитали обратно.**

In [ ]:
tmp_csv = paths.STAGING / "_opyt.csv"
tmp_parquet = paths.STAGING / "_opyt.parquet"

sample = orders.head(1000).copy()
sample["order_purchase_timestamp"] = pd.to_datetime(sample["order_purchase_timestamp"])

sample.to_csv(tmp_csv, index=False)
sample.to_parquet(tmp_parquet, index=False)

print("исходный тип даты        :", sample["order_purchase_timestamp"].dtype)
print("после записи в CSV       :",
      pd.read_csv(tmp_csv)["order_purchase_timestamp"].dtype, " – тип потерян")
print("после записи в Parquet   :",
      pd.read_parquet(tmp_parquet)["order_purchase_timestamp"].dtype, " – тип сохранён")
print()
print(f"объём CSV     : {tmp_csv.stat().st_size / 1024:7.1f} КБ")
print(f"объём Parquet : {tmp_parquet.stat().st_size / 1024:7.1f} КБ")

Мы привели дату к нормальному типу, записали результат – и следующий читатель
получит снова текст. Причина не в pandas: в формате CSV негде хранить описание
колонок. Отсюда расхождение, когда у аналитика выручка одна, а у инженера
другая, хотя файл читали один и тот же.

**Запустили обработку дважды.**

In [ ]:
append_path = paths.STAGING / "_opyt_append.csv"
append_path.unlink(missing_ok=True)

for run in (1, 2):
    demo.to_csv(append_path, mode="a", index=False, header=(run == 1))
    print(f"после запуска {run}: строк в файле {len(pd.read_csv(append_path)):>4}")

print(f"\nстрок должно быть {len(demo)}")
print("Запись с заменой файла дала бы один и тот же результат при любом числе запусков.")

Разница между «дописать» и «заменить» выглядит придиркой ровно до первого
ночного сбоя. Обработка падает и перезапускается регулярно: пропал доступ к источнику,
кончилось место, кто-то нажал повтор. Свойство операции давать один и тот же
результат при любом числе повторов называется **идемпотентностью
(idempotency)**, и в пятой работе оно станет обязательным для каждого шага.

## Задание

Аналитик уточнил, что именно ему нужно. Постройте таблицу `daily` – одна строка
на календарный день покупки:

| колонка | смысл |
|---|---|
| `order_date` | дата покупки из `order_purchase_timestamp` |
| `n_orders` | сколько заказов сделано в этот день |
| `n_items` | сколько товарных позиций в этих заказах |
| `revenue` | сумма `price` по позициям |
| `freight` | сумма `freight_value` по позициям |

Условия, за которые отвечаете вы:

1. Учитываем все заказы, без отбора по статусу.
2. Число заказов за каждый день обязано совпасть с числом заказов этого дня
   в исходной таблице. Добавление второго источника не имеет права изменить
   количество заказов – это главная проверка работы.
3. В данных есть заказы вообще без позиций: заказ отменили до сборки. Они
   должны остаться в отчёте с нулями, а не исчезнуть.
4. Количество – целое число, деньги – дробное.

Прежде чем писать код, ответьте себе на три вопроса: до какого зерна нужно
свернуть позиции, чтобы соединение не размножило заказы; какое соединение
сохранит заказы без позиций; что честнее поставить в выручку такого заказа –
пропуск или ноль.

Решение может быть не одно: можно свернуть позиции до заказа и потом соединить,
можно посчитать две независимые сводки по дате и соединить их. Проверка смотрит
на свойства результата, а не на выбранный путь.

In [ ]:
# Шаг 1. Привести дату покупки к нормальному типу и выделить календарный день.
# Шаг 2. Решить, как соединить заказы и позиции, не размножив заказы.
# Шаг 3. Свернуть до дня и посчитать четыре показателя.
# Шаг 4. Заполнить нулями там, где по смыслу ноль, а не пропуск.

daily = None

if daily is None:
    print("таблица daily ещё не построена")
else:
    display(daily.head())

In [ ]:
from checks.lab01 import check_task_1, lab01_result

out_path = paths.STAGING / "lab01_daily_orders.parquet"
if daily is not None:
    load_parquet(daily, out_path)
    print("сохранено:", out_path.relative_to(paths.ROOT))

check_task_1(daily, orders, items)

## Та же задача в другой системе

Интернет-магазин мебели, две просьбы от бизнеса.

Первая. Каждое утро к девяти директор хочет видеть отчёт за вчера: выручка,
число заказов, средний чек, десять самых ходовых категорий. Данные лежат в базе
магазина, прирост – около 30 МБ в сутки.

Вторая. На странице товара нужен блок «пять человек смотрят прямо сейчас», а
если покупатель положил товар в корзину и ушёл, через двадцать минут ему нужно
отправить напоминание. Событий с сайта около пятисот в секунду в пике.

Ответьте письменно по каждой просьбе: какие здесь источники; пакетная обработка
или потоковая и почему; где хранить данные; какие преобразования нужны; что
может испортить цифры; какие части системы нужны; какие были бы здесь лишними.

Подсказка о способе рассуждения: ответ на вопрос «нужен ли брокер событий»
звучит не «да, это же события», а «вот допустимая задержка, вот что сломается
при её превышении».

### Ваш ответ

**Утренний отчёт.**

1. Источники: …
2. Пакетная обработка или потоковая: …
3. Хранение: …
4. Преобразования: …
5. Риски для цифр: …
6. Нужные части системы: …
7. Лишние части: …

**События сайта.**

1. Источники: …
2. Пакетная обработка или потоковая: …
3. Хранение: …
4. Преобразования: …
5. Риски для цифр: …
6. Нужные части системы: …
7. Лишние части: …

In [ ]:
lab01_result()

## Вопросы для проверки себя

1. Вы соединили заказы с платежами и посчитали выручку как сумму `payment_value`.
   Число строк выросло. В каком случае сумма останется правильной, а в каком
   нет? Ответ объясните через зерно таблиц.
2. Почему соединение из предыдущего вопроса технически корректно, но результат
   при этом может быть неверным?
3. Зачем хранить сырой слой отдельно, если данные обычно можно выгрузить из
   источника заново? Приведите два случая, когда «выгрузить заново» не сработает.
4. У вас 30 МБ новых данных в сутки и требование «отчёт к утру». Нужна ли здесь
   потоковая обработка? Что должно измениться в требованиях, чтобы ответ стал
   другим?
5. Скрипт дважды дописал результат в файл, данные задвоились. Назовите два
   разных способа сделать этот шаг безопасным при повторном запуске.

## Что дальше

У нас появился первый путь от файлов до отчёта, а у каждой таблицы – понятный
смысл строки. Но данные по-прежнему читаются в память целиком, типы теряются
при каждой записи, а весь расчёт существует в виде кода, который понимает
только его автор. Аналитику, которому нужно посчитать что-то своё, к этому коду
не подойти.

В следующей работе положим данные в хранилище, которое помнит свою схему,
и начнём считать на SQL.